# 09 · Explore CellBridge predictions in tissue

A visual workbench for discussing **fine-grained single-cell phenotyping** with biological collaborators. Start with **NuCLS** or **PhenoCell**; HEST sections are optional. All plots call the model **CellBridge** (all training cells, adapted representation, last four transformer blocks adapted, linear classifier). This notebook reads the saved conformal evaluation; it does not train a model or change calibration.

**How to use:** run all cells once, choose a dataset below, and click **Load dataset**. Choose a field, then a patch or a cell, and click **Show selection**. Every view uses the branch-graph conformal method. A later section maps whole HEST sections patch by patch and opens any patch in that same view; the last section projects a HEST patch into its TERRA transcriptomic embeddings. The setup/helper cells are collapsed in supporting notebook viewers.

**Environment:** use the lab's `hest` Python kernel, which contains the image readers and widgets. To start Jupyter from a terminal:
```bash
module load ISG/conda
conda activate hest
cd /lustre/scratch126/cellgen/lotfollahi/so25/projects/cellbridge
PYTHONPATH=src jupyter lab notebooks/09_qualitative_conformal.ipynb
```
The notebook needs this checkout, prepared datasets and saved results on the lab filesystem; it needs neither a GPU nor the sandbox scripts. Paths below respect `CELLBRIDGE_DATA_ROOT` and `CELLBRIDGE_QUALITATIVE_RESULTS` for another machine.

## What am I looking at?

| Term | Meaning |
|---|---|
| Ground truth | The dataset's reference annotation, harmonized to the evaluated cell classes. It is a reference label, not an independent biological verification. The original annotation is also shown. |
| Predicted label | CellBridge's highest-scoring class, shared by all conformal methods. |
| Prediction set | The candidate labels stored by the selected conformal method. An empty set means no label was returned. |
| Lowest common ancestor (LCA) | The most specific common parent of the candidate labels in the Cell Ontology. For example, distinct lymphocyte types can share a broader lymphocyte ancestor. |
| Ancestor-expanded set | All **evaluated classes** under that ancestor. This may add labels absent from the original set; its coverage is reported separately. |
| Depth / coarseness | Number of steps **up from the ground-truth ontology term to the LCA**, along the shortest path. **0 = same term; larger = less specific.** This is the retrospective uncertainty convention in the ancestor-coarseness script, not depth measured down from the ontology root and not a confidence probability. |

A black **×** means the ancestor excludes the reference label; no depth is assigned. An empty set has a red **+**. Cells without a saved test prediction have hollow grey markers and no uncertainty score. These markers represent annotated cell/nucleus centres, not segmentation boundaries or every visible cell.

The ontology is a graph: a type can have more than one parent and sometimes more than one LCA. We list all lowest candidates and choose one deterministically, following the supplied ancestor script: fewest descendant classes, then greatest root depth, then term ID. Synthetic classifier-leaf edges are excluded from the reported depth. A singleton retains its exact classifier class when forming the expanded set.

The saved nominal coverage is a population-level target. Observed coverage in a selected patch can differ substantially; a selected example is not a benchmark estimate. Smoothing, when enabled, shows a local mean among ancestor-covered cells only and cannot repair excluded cells.

In [ ]:
import json
import os
import pickle
import sys
from collections import deque
from functools import lru_cache
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, clear_output, display
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.lines import Line2D
from PIL import Image

# Find the checkout when launched from the repository, notebooks/, or workspace.
REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / "projects/cellbridge")
             if (p / "src/cellbridge/config.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Launch Jupyter from the cellbridge checkout.")
sys.path.insert(0, str(REPO / "src"))
from cellbridge.config import load_config
from cellbridge.inference.conformal.graph import build_task_graph
from cellbridge.inference.embed.sources import hest, lmdb as tiled, nucls
from cellbridge.ontology import load_mapping, load_ontology

WORKSPACE = Path(os.environ.get("CELLBRIDGE_WORKSPACE", REPO.parents[1]))
MODEL_RESULTS = Path(os.environ.get("CELLBRIDGE_QUALITATIVE_RESULTS",
    WORKSPACE / "outputs/ablations/lora_depth/layers_4_cells_all/inference"))
TRAIN_CONFIG = MODEL_RESULTS.parent / "train/resolved_config.yaml"
training = load_config(TRAIN_CONFIG)
assert training["model"]["trainable_blocks"] == "last_4", "Expected four adapted blocks"
assert training["max_cells_per_sample"] is None, "Expected training on all cells"
EXPORT_ROOT = WORKSPACE / "outputs/reproducibility/figures/qualitative_conformal"
METHODS = {
    "split": "Global split", "graph": "Global graph",
    "class_split": "Class split", "class_graph": "Class graph",
    "branch_split": "Branch split", "branch_graph": "Branch graph",
    "hybrid_split": "Hybrid split", "hybrid_graph": "Hybrid graph",
}
# The views offer one conformal method. Branch graph is the reported method, and a fixed
# choice keeps every view comparable; METHODS stays complete so the saved metrics of all
# eight are still checked when a dataset loads.
OFFERED_METHODS = ["branch_graph"]
plt.rcParams.update({"text.usetex": False, "font.family": "DejaVu Sans", "font.size": 10})
print("CellBridge results:", MODEL_RESULTS)

## Available datasets and sections

Only entries with saved cell-level predictions appear. HEST remains exploratory; TENX143 is omitted to match notebook 08. The default field has the most evaluated cells (ties use field ID), **not** the best performance. Field counts and full-dataset method metrics appear after loading. NuCLS has fewer evaluated classes than original annotation types because the saved evaluation filtered classes by support.

In [ ]:
def prediction_directory(dataset, section=None):
    base = MODEL_RESULTS / dataset
    if section:
        return base / section / "evaluation/lora_adapted/all/linear" / section / "conformal_prediction"
    return base / "evaluation/lora_adapted/all/linear/conformal_prediction"

# HEST organs come from the phenotype cohort table. The two cell-level datasets carry no
# organ column: NuCLS is TCGA breast carcinoma, PhenoCell the Schürch colorectal cohort.
DATASET_ORGANS = {"nucls": "breast", "phenocell": "colorectal"}
cohort = pd.read_csv(load_config(REPO / "configs/inference/hest.yaml")["data"]["cohort"],
                     dtype={"sample_id": str})
HEST_ORGANS = dict(zip(cohort.sample_id, cohort.organ.str.replace("_", " ")))

catalog = []
for dataset, label in [("nucls", "NuCLS"), ("phenocell", "PhenoCell")]:
    directory = prediction_directory(dataset)
    if (directory / "predictions.npz").is_file():
        organ = DATASET_ORGANS[dataset]
        catalog.append(dict(choice=f"{label} ({organ})", dataset=dataset, section=None,
                            organ=organ, directory=directory))
for path in sorted((MODEL_RESULTS / "hest").glob(
        "*/evaluation/lora_adapted/all/linear/*/conformal_prediction/predictions.npz")):
    section = path.parent.parent.name
    if section != "TENX143" and path == prediction_directory("hest", section) / "predictions.npz":
        organ = HEST_ORGANS.get(section, "organ not recorded")
        catalog.append(dict(choice=f"HEST · {section} ({organ})", dataset="hest",
                            section=section, organ=organ, directory=path.parent))
if not catalog:
    raise FileNotFoundError(f"No CellBridge cell-level conformal results under {MODEL_RESULTS}")
catalog = pd.DataFrame(catalog).set_index("choice")
display(catalog[["dataset", "section", "organ"]].fillna("—"))

## Data and ontology helpers

Run these cells once. They join predictions by stable cell ID, check labels against the dataset mapping, and reproduce saved coverage/set-size metrics before displaying tissue. PhenoCell is reconstructed from its original padded tiles; HEST reads only the selected region of its whole-slide image.

In [ ]:
def load_dataset(choice):
    spec = catalog.loc[choice]
    dataset, section, directory = spec.dataset, spec.section, spec.directory
    config = load_config(REPO / f"configs/inference/{dataset}.yaml")
    data = config["data"]
    mapping_path = (Path(data["mapping_dir"]) / f"{section}.yaml" if dataset == "hest"
                    else Path(config["conformal"]["mapping"]))
    mapping = load_mapping(mapping_path)
    ontology = load_ontology(config["conformal"]["ontology"])
    with np.load(directory / "predictions.npz", allow_pickle=False) as archive:
        saved = {key: archive[key] for key in archive.files}
    ids, classes = saved["ids"].astype(str), saved["classes"].astype(str)
    assert pd.Index(ids).is_unique and pd.Index(classes).is_unique
    available = [m for m in METHODS if m + "_sets" in saved]
    metrics = pd.read_csv(directory / "metrics.csv").set_index("conformal_method", verify_integrity=True)
    assert metrics["method"].eq("cellbridge").all()
    assert metrics["arm"].eq("lora_adapted").all()
    assert metrics["task"].eq("all").all() and metrics["probe"].eq("linear").all()
    sample = None
    if dataset == "hest":
        sample = hest.load_sample(data["manifest"], data["root"], section)
        labels_path, _ = hest.evaluation_paths(data, section)
        labels = pd.read_csv(labels_path, dtype={"cell_id": str}).rename(columns={"group": "label"})
        metadata = pd.DataFrame({"row_id": sample.cell_ids.astype(str), "sample_id": section,
                                 "x": sample.centers[:, 0], "y": sample.centers[:, 1]})
        metadata = metadata.merge(labels[["cell_id", "label"]], left_on="row_id", right_on="cell_id",
                                  how="left", validate="one_to_one")
        metadata["label"] = metadata["label"].fillna("No reference annotation")
    else:
        # Include all available source annotations in overlays, even excluded classes.
        source_data = dict(data, exclude_labels=[])
        metadata = (nucls if dataset == "nucls" else tiled).load_metadata(source_data)
    metadata["row_id"] = metadata["row_id"].astype(str)
    lookup = metadata.set_index("row_id", drop=False, verify_integrity=True)
    evaluated = lookup.loc[ids].copy()
    if dataset != "hest":
        assert evaluated["split"].eq("test").all()
    truth, included = mapping.map_labels(evaluated["label"].astype(str).to_numpy())
    assert included.all()
    np.testing.assert_array_equal(truth, saved["y_true"].astype(str))
    columns = pd.Index(classes).get_indexer(truth)
    assert (columns >= 0).all()
    probabilities = saved["probabilities"]
    assert probabilities.shape == (len(ids), len(classes)) and np.isfinite(probabilities).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    for method in available:
        masks = saved[method + "_sets"]
        assert masks.dtype == bool and masks.shape == probabilities.shape
        actual = [masks[np.arange(len(ids)), columns].mean(), masks.sum(axis=1).mean(),
                  (masks.sum(axis=1) == 1).mean()]
        expected = metrics.loc[method, ["marginal_coverage", "mean_prediction_set_size", "singleton_rate"]]
        np.testing.assert_allclose(actual, expected.to_numpy(float))
    evaluated["truth"] = truth
    evaluated["predicted"] = classes[probabilities.argmax(axis=1)]
    graph = build_task_graph(mapping, ontology, classes)
    state = dict(choice=choice, dataset=dataset, section=section, config=config, mapping=mapping,
                 mapping_path=mapping_path, ontology=ontology, graph=graph, saved=saved,
                 metadata=metadata, evaluated=evaluated, methods=available, metrics=metrics,
                 directory=directory, sample=sample)
    counts = evaluated.groupby("sample_id").size().rename("evaluated_cells").to_frame()
    counts["annotated_cells"] = metadata.groupby("sample_id").size()
    counts["top1_accuracy"] = evaluated.groupby("sample_id").apply(
        lambda rows: rows.truth.eq(rows.predicted).mean())
    state["fields"] = counts.reset_index().sort_values(
        ["evaluated_cells", "sample_id"], ascending=[False, True])
    return state


def load_field(state, sample_id):
    data = state["config"]["data"]
    cells = state["metadata"].loc[state["metadata"].sample_id.eq(sample_id)].copy()
    if state["dataset"] == "nucls":
        names = cells.image_name.unique()
        assert len(names) == 1
        path = Path(data["images"]) / names[0]
        with Image.open(path) as image:
            tissue = np.asarray(image.convert("RGB"))
        mpp = data["source_mpp"]
    elif state["dataset"] == "phenocell":
        import lmdb
        print("PhenoCell: reading original tiles; the first load can take several minutes.")
        # This dataset stores a fixed block of consecutive keys for every field.
        n_tiles = int(np.prod(np.ceil(np.asarray(data["image_shape"]) / data["tile_size"])))
        with lmdb.open(str(data["lmdb"]), readonly=True, lock=False, readahead=True) as env:
            assert env.stat()["entries"] == state["metadata"].sample_id.nunique() * n_tiles
            entries = None
            # Metadata order supplies a fast candidate; verify the stored name.
            # Fall back to scanning block headers if that order ever changes.
            sample_order = state["metadata"].sample_id.drop_duplicates().tolist()
            guess = sample_order.index(sample_id) * n_tiles
            starts = dict.fromkeys([guess, *range(0, env.stat()["entries"], n_tiles)])
            with env.begin() as txn:
                for start in starts:
                    first = pickle.loads(txn.get(f"{start:08d}".encode()))
                    if str(first["sample_name"]) != sample_id:
                        continue
                    entries = {}
                    for key in range(start, start + n_tiles):
                        item = pickle.loads(txn.get(f"{key:08d}".encode()))
                        assert str(item["sample_name"]) == sample_id
                        index = tiled._tile_index(item["tile_name"])
                        assert index not in entries
                        entries[index] = item
                    break
        if entries is None:
            raise ValueError(f"Field {sample_id} is absent from the PhenoCell LMDB")
        sample = tiled._load_sample(sample_id, entries, cells, image_shape=data["image_shape"],
                                    tile_size=data["tile_size"], value_scale=data["value_scale"],
                                    skip_missing=False)
        tissue, cells, mpp = sample.image, sample.cells, data["source_mpp"]
        path = Path(data["lmdb"])
    else:
        sample = state["sample"]
        tissue, mpp, path = hest.open_wsi(sample.wsi_path), sample.pixel_size_um, sample.wsi_path
    height, width = tissue.shape[:2]
    inside = (cells.x >= 0) & (cells.x < width) & (cells.y >= 0) & (cells.y < height)
    omitted = int((~inside).sum())
    omitted_evaluated = int(cells.loc[~inside, "row_id"].isin(state["evaluated"].index).sum())
    if state["dataset"] != "hest":
        assert inside.all(), "Source coordinates lie outside the reconstructed image"
    cells = cells.loc[inside].copy()
    if omitted:
        print(f"Omitted {omitted} annotations outside H&E bounds ({omitted_evaluated} evaluated).")
    cells = cells.sort_values("row_id").reset_index(drop=True)
    cells["number"] = np.arange(1, len(cells) + 1)
    cells["evaluated"] = cells.row_id.isin(state["evaluated"].index)
    return dict(sample_id=sample_id, tissue=tissue, cells=cells, mpp=float(mpp), image_path=str(path),
                outside_image_annotations=omitted, outside_image_evaluated=omitted_evaluated)


def ancestor_functions(state):
    graph, ontology = state["graph"], state["ontology"]
    @lru_cache(None)
    def root_depth(node):
        return max((root_depth(p) + 1 for p in graph.parents[node]), default=0)
    @lru_cache(None)
    def close(columns):
        if not columns:
            return "", (), (), ""
        candidates = {node for node, descendants in graph.descendant_columns.items()
                      if set(columns).issubset(descendants)}
        lowest = [n for n in candidates if not candidates.intersection(graph.children[n])]
        lowest.sort(key=lambda n: (len(graph.descendant_columns[n]), -root_depth(n), n))
        anchor = lowest[0]
        term = graph.parents[anchor][0] if anchor in graph.class_nodes else anchor
        terms = tuple(graph.parents[n][0] if n in graph.class_nodes else n for n in lowest)
        return anchor, graph.descendant_columns[anchor], terms, term
    return close


def evaluate_field(state, field, method):
    graph, classes = state["graph"], state["saved"]["classes"].astype(str)
    cells = field["cells"].copy()
    selected = cells.loc[cells.evaluated]
    positions = pd.Index(state["saved"]["ids"].astype(str)).get_indexer(selected.row_id)
    assert (positions >= 0).all()
    close = ancestor_functions(state)
    records = []
    for row_id, position in zip(selected.row_id, positions):
        mask = state["saved"][method + "_sets"][position]
        columns = tuple(np.flatnonzero(mask))
        anchor, expanded, lowest, term = close(columns)
        truth = str(state["saved"]["y_true"][position])
        true_column = int(np.flatnonzero(classes == truth)[0])
        covered = true_column in expanded
        distance = max(0, graph.ancestors_for_column(true_column)[anchor] - 1) if covered else np.nan
        records.append(dict(row_id=row_id, truth=truth,
            predicted=classes[state["saved"]["probabilities"][position].argmax()],
            prediction_set="; ".join(classes[list(columns)]) or "∅", set_size=len(columns),
            raw_covered=bool(mask[true_column]), ancestor_term=term,
            ancestor=state["ontology"].names[term] if term else "Empty set",
            all_lowest_ancestors="; ".join(f"{state['ontology'].names[t]} ({t})" for t in lowest),
            expanded_set="; ".join(classes[list(expanded)]) or "∅",
            ancestor_covered=covered, depth=distance))
    return cells.merge(pd.DataFrame(records), on="row_id", how="left", validate="one_to_one")

In [ ]:
# One fixed depth colour ramp, shared by the per-cell, smoothed and patch maps.
DEPTH_CMAP = LinearSegmentedColormap.from_list(
    "ontology_depth", ["#4055cf", "#63d6e5", "#c5f0b1", "#fff28a", "#f6a15c", "#bf3438"]
)


def patch_choices(field, size=512):
    cells = field["cells"].loc[field["cells"].evaluated]
    bins = pd.DataFrame({"column": (cells.x // size).astype(int), "row": (cells.y // size).astype(int)})
    counts = bins.groupby(["row", "column"]).size().sort_index()
    height, width = field["tissue"].shape[:2]
    return [(f"x={c * size}, y={r * size} · {n} evaluated cells",
             (c * size, r * size, min((c + 1) * size, width), min((r + 1) * size, height)))
            for (r, c), n in counts.items()]


def cells_in_bounds(cells, bounds):
    x0, y0, x1, y1 = bounds
    return cells.loc[(cells.x >= x0) & (cells.x < x1) & (cells.y >= y0) & (cells.y < y1)].copy()


def smooth_depth(tissue, rows, mpp, bounds):
    from scipy.ndimage import gaussian_filter
    from scipy.spatial import cKDTree
    x0, y0, x1, y1 = bounds
    step, bandwidth_um = 4, 15.0
    covered = rows.loc[rows.depth.notna()]
    if covered.empty:
        return None
    xe, ye = np.arange(x0, x1 + step, step), np.arange(y0, y1 + step, step)
    counts, _, _ = np.histogram2d(covered.y, covered.x, bins=(ye, xe))
    totals, _, _ = np.histogram2d(covered.y, covered.x, bins=(ye, xe), weights=covered.depth)
    sigma = bandwidth_um / (mpp * step)
    density = gaussian_filter(counts, sigma, mode="constant", truncate=3)
    numerator = gaussian_filter(totals, sigma, mode="constant", truncate=3)
    mean = np.divide(numerator, density, out=np.full_like(density, np.nan), where=density > 0)
    xx, yy = np.meshgrid((xe[:-1] + xe[1:]) / 2, (ye[:-1] + ye[1:]) / 2)
    near, _ = cKDTree(covered[["x", "y"]]).query(np.column_stack([xx.ravel(), yy.ravel()]))
    rgb = tissue[(yy - y0).astype(int).clip(0, tissue.shape[0] - 1),
                 (xx - x0).astype(int).clip(0, tissue.shape[1] - 1)].astype(float)
    sat = (rgb.max(axis=2) - rgb.min(axis=2)) / np.maximum(rgb.max(axis=2), 1)
    foreground = gaussian_filter((sat > 0.06).astype(float), 1) > 0.5
    valid = foreground & (near.reshape(mean.shape) * mpp <= 30) & (density * 2 * np.pi * sigma**2 >= 3)
    mean[~valid] = np.nan
    return mean, (xe[0], xe[-1], ye[-1], ye[0])


def show_table(table):
    display(HTML('<div style="max-height:420px;overflow:auto">' +
                 table.to_html(index=False, escape=True, na_rep="—") + '</div>'))


def shortest_path(ontology, start, target):
    pending = deque([(start,)])
    seen = {start}
    while pending:
        path = pending.popleft()
        if path[-1] == target:
            return path
        for parent in sorted(ontology.parents[path[-1]]):
            if parent not in seen:
                seen.add(parent)
                pending.append((*path, parent))
    return ()


def render_selection(state, field, method, bounds, cell_id=None, smooth=False):
    all_rows = evaluate_field(state, field, method)
    rows = cells_in_bounds(all_rows, bounds)
    evaluated = rows.loc[rows.evaluated]
    x0, y0, x1, y1 = map(int, bounds)
    tissue = np.asarray(field["tissue"][y0:y1, x0:x1])[..., :3]
    extent = (x0 - .5, x1 - .5, y1 - .5, y0 - .5)
    classes = state["saved"]["classes"].astype(str)
    colours = dict(zip(classes, plt.get_cmap("tab20")(np.linspace(0, 1, len(classes)))))
    graph = state["graph"]
    # One fixed scale for every method and field within the dataset/section.
    maximum = max(max(distances.values()) - 1 for distances in graph.distances)
    cmap = ListedColormap(DEPTH_CMAP(np.linspace(0, 1, maximum + 1)))
    norm = BoundaryNorm(np.arange(-.5, maximum + 1.5), cmap.N)
    fig, axes = plt.subplots(2, 2, figsize=(15, 12), layout="constrained")
    for ax in axes.flat:
        ax.imshow(tissue, extent=extent, alpha=1 if ax is axes[0, 0] else .65)
        ax.set(xlim=(x0 - .5, x1 - .5), ylim=(y1 - .5, y0 - .5))
        ax.axis("off")
    axes[0, 0].set_title("H&E · selected cell circled when applicable")
    axes[0, 1].set_title("Reference labels · numbers match the table")
    axes[1, 0].set_title("CellBridge · highest-scoring label")
    axes[1, 1].set_title("Ontology depth / coarseness" + (" · local mean" if smooth else " · per cell"))
    for ax, column in [(axes[0, 1], "truth"), (axes[1, 0], "predicted")]:
        for label in classes:
            group = evaluated.loc[evaluated[column].eq(label)]
            ax.scatter(group.x, group.y, s=50, color=colours[label], edgecolors="white", linewidths=.4)
    missing = rows.loc[~rows.evaluated]
    # Harmonized reference labels are still visible for non-evaluated cells, when available.
    reference = missing.label.map(state["mapping"].source_to_class)
    for label in classes:
        group = missing.loc[reference.eq(label)]
        axes[0, 1].scatter(group.x, group.y, s=54, facecolors="none", edgecolors=[colours[label]])
    unknown = missing.loc[~reference.isin(classes)]
    axes[0, 1].scatter(unknown.x, unknown.y, s=54, facecolors="none", edgecolors="grey")
    for ax in axes[1]:
        ax.scatter(missing.x, missing.y, s=54, facecolors="none", edgecolors="grey")
    covered = evaluated.loc[evaluated.depth.notna()]
    if smooth:
        smoothed = smooth_depth(tissue, rows, field["mpp"], bounds)
        if smoothed is not None and np.isfinite(smoothed[0]).any():
            axes[1, 1].imshow(smoothed[0], extent=smoothed[1], cmap=DEPTH_CMAP, vmin=0, vmax=maximum,
                              alpha=.78, interpolation="nearest")
        else:
            axes[1, 1].text(.5, .03, "No supported neighbourhoods to smooth", ha="center",
                            transform=axes[1, 1].transAxes, bbox=dict(facecolor="white", alpha=.9))
    else:
        axes[1, 1].scatter(covered.x, covered.y, c=covered.depth, cmap=cmap, norm=norm,
                          s=64, edgecolors="white", linewidths=.4)
    excluded = evaluated.loc[evaluated.depth.isna() & evaluated.set_size.gt(0)]
    empty = evaluated.loc[evaluated.set_size.eq(0)]
    axes[1, 1].scatter(excluded.x, excluded.y, marker="x", c="black", s=80)
    axes[1, 1].scatter(empty.x, empty.y, marker="+", c="#c62828", s=110)
    bar = plt.cm.ScalarMappable(cmap=DEPTH_CMAP if smooth else cmap,
                               norm=plt.Normalize(0, maximum) if smooth else norm)
    fig.colorbar(bar, ax=axes[1, 1], shrink=.7, ticks=range(maximum + 1), label="Steps up from reference to ancestor")
    if len(rows) <= 60:
        for row in rows.itertuples():
            axes[0, 1].annotate(str(row.number), (row.x, row.y), xytext=(3, 3),
                                 textcoords="offset points", fontsize=7,
                                 bbox=dict(facecolor="white", alpha=.7, edgecolor="none", pad=.2))
    if cell_id is not None:
        chosen = rows.loc[rows.row_id.eq(cell_id)].iloc[0]
        for ax in axes.flat:
            ax.scatter([chosen.x], [chosen.y], s=520, facecolors="none", edgecolors="magenta", linewidths=2)
    scale_um = 10 ** np.floor(np.log10((x1 - x0) * field["mpp"] / 4))
    bx, by = x0 + .06 * (x1 - x0), y1 - .07 * (y1 - y0)
    axes[0, 0].plot([bx, bx + scale_um / field["mpp"]], [by, by], c="black", lw=3)
    axes[0, 0].text(bx, by - .025 * (y1 - y0), f"{scale_um:g} µm",
                      bbox=dict(facecolor="white", alpha=.8, edgecolor="none"))
    handles = [Line2D([], [], marker="o", linestyle="none", color=colours[c], label=c) for c in classes]
    axes[0, 1].legend(handles=handles, loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
    axes[1, 1].legend(handles=[
        Line2D([], [], marker="x", color="black", linestyle="none", label="Reference excluded by ancestor"),
        Line2D([], [], marker="+", color="#c62828", linestyle="none", label="Empty prediction set"),
        Line2D([], [], marker="o", markerfacecolor="none", color="grey", linestyle="none", label="No test prediction")],
        loc="upper center", bbox_to_anchor=(.5, 0), fontsize=8)
    fig.suptitle(f"{state['choice']} · CellBridge · {METHODS[method]}\n"
                 f"{field['sample_id']}\nx={x0}:{x1}, y={y0}:{y1}", fontsize=11, wrap=True)
    display(fig)
    plt.close(fig)
    nominal = float(state["metrics"].loc[method, "target_coverage"])
    display(Markdown(f"**{len(rows):,} annotated cells; {len(evaluated):,} with test predictions.** "
        f"Saved nominal coverage: {nominal:.0%}. Numbers are omitted above 60 cells; choose a smaller patch or cell view."))
    if len(evaluated):
        show_table(pd.DataFrame([{
            "top-1 correct": evaluated.truth.eq(evaluated.predicted).mean(),
            "original-set coverage": evaluated.raw_covered.astype(bool).mean(),
            "ancestor-expanded coverage": evaluated.ancestor_covered.astype(bool).mean(),
            "mean set size": evaluated.set_size.mean(), "mean depth (covered only)": evaluated.depth.mean(),
            "empty sets": int(evaluated.set_size.eq(0).sum())}]))
    columns = ["number", "row_id", "x", "y", "label", "evaluated", "truth", "predicted", "prediction_set",
               "set_size", "raw_covered", "ancestor", "ancestor_term", "all_lowest_ancestors",
               "expanded_set", "ancestor_covered", "depth"]
    if cell_id is not None:
        display(Markdown("**Selected cell** (circled in magenta):"))
        details = rows.loc[rows.row_id.eq(cell_id), columns].iloc[0]
        show_table(details.rename_axis("item").reset_index(name="value"))
    display(Markdown("**All annotated cells in this view:**"))
    show_table(rows[columns])
    if cell_id is not None:
        chosen = rows.loc[rows.row_id.eq(cell_id)].iloc[0]
        if chosen.evaluated and chosen.ancestor_term:
            labels = chosen.prediction_set.split("; ")
            terms = state["mapping"].class_terms(labels)
            paths = []
            for label, term in terms.items():
                path = shortest_path(state["ontology"], term, chosen.ancestor_term)
                paths.append({"candidate label": label, "path up to common ancestor":
                              " → ".join(f"{state['ontology'].names[t]} ({t})" for t in path)})
            display(Markdown("**Selected cell: candidate paths to the ancestor** (one shortest path per label)."))
            show_table(pd.DataFrame(paths))
    return dict(figure=fig, cells=rows[columns], method=method, bounds=list(map(int, bounds)),
                cell_id=cell_id, smooth=smooth, sample_id=field["sample_id"], image_path=field["image_path"],
                mpp=field["mpp"], outside_image_annotations=field["outside_image_annotations"],
                outside_image_evaluated=field["outside_image_evaluated"])

## Explore

1. **Load dataset** lists its fields and validates its saved predictions. HEST section IDs are in the dataset menu.
2. Choose a **field**, then a **patch**. Grid coordinates are in original H&E pixels, starting at the top left. Only grid patches containing evaluated cells are listed. A PhenoCell field is reconstructed before it can be browsed; the first load from the shared filesystem can take several minutes.
3. Choose **Patch** to see the patch, or **Cell** to centre a neighbourhood on a cell from it. The cell list includes the plot number, stable row ID and original annotation. Enable **Cells from whole field** to find a cell outside the selected patch. **Find cell** filters by row ID or original annotation; suggestions are limited to 500 so large HEST sections remain responsive.
4. Click **Show selection**. **Save displayed view** exports the last rendered figure, complete cell table and provenance; merely changing a control does not change the saved view.

**Method** is fixed to **branch graph**: candidate sets are calibrated within groups of related cell types and are formed along the ontology rather than by thresholding scores alone. The saved evaluation holds seven further methods and this notebook still checks all of their stored metrics when a dataset loads, but only branch graph is offered here, so every view is directly comparable. [Notebook 08](08_conformal_prediction.ipynb) compares the methods against each other.

Optional smoothing uses a 15 µm Gaussian bandwidth, masks unstained tissue, and requires a nearby covered cell (≤30 µm) and weighted support of at least three cells. It is calculated within the displayed crop, so values near its boundary can change when the crop changes. Colours use a fixed scale across methods and fields within a dataset/section; ontology depth is not normalized between datasets.

In [ ]:
dataset_widget = widgets.Dropdown(options=list(catalog.index), description="Dataset:", layout=widgets.Layout(width="450px"))
load_button = widgets.Button(description="Load dataset", button_style="primary")
field_widget = widgets.Dropdown(description="Field:", layout=widgets.Layout(width="95%"))
patch_widget = widgets.Dropdown(description="Patch:", layout=widgets.Layout(width="650px"))
mode_widget = widgets.ToggleButtons(options=["Patch", "Cell"], description="View:")
size_widget = widgets.Dropdown(options=[224, 512, 1024], value=512, description="Width (px):")
whole_field_widget = widgets.Checkbox(value=False, description="Cells from whole field")
cell_search = widgets.Text(description="Find cell:", placeholder="Optional row ID or original annotation",
                           layout=widgets.Layout(width="650px"))
cell_count = widgets.Label()
cell_widget = widgets.Combobox(options=[], ensure_option=True, description="Cell:", layout=widgets.Layout(width="95%"))
method_widget = widgets.Dropdown(description="Method:", layout=widgets.Layout(width="350px"))
smooth_widget = widgets.Checkbox(value=False, description="Smooth depth map")
show_button = widgets.Button(description="Show selection", button_style="primary", disabled=True)
save_button = widgets.Button(description="Save displayed view", disabled=True)
status_output, catalog_output, plot_output = widgets.Output(), widgets.Output(), widgets.Output()
browser = {"state": None, "field": None, "last": None, "cell_ids": {}}


def update_cells(*_):
    field = browser["field"]
    if field is None or patch_widget.value is None:
        return
    rows = field["cells"] if whole_field_widget.value else cells_in_bounds(field["cells"], patch_widget.value)
    query = cell_search.value.strip().lower()
    if query:
        rows = rows.loc[rows.row_id.str.lower().str.contains(query, regex=False) |
                        rows.label.str.lower().str.contains(query, regex=False)]
    cell_count.value = f"{len(rows):,} matching cells; showing up to 500 suggestions. Refine Find cell if needed."
    rows = rows.head(500)
    browser["cell_ids"] = {
        f"#{r.number} | {r.row_id} | {r.label}" + ("" if r.evaluated else " [no test prediction]"): r.row_id
        for r in rows.itertuples()}
    cell_widget.options = list(browser["cell_ids"])
    evaluated_ids = set(rows.loc[rows.evaluated, "row_id"])
    cell_widget.value = next((label for label, uid in browser["cell_ids"].items() if uid in evaluated_ids),
                             cell_widget.options[0] if cell_widget.options else "")


def update_patches(*_):
    if browser["field"] is not None:
        patch_widget.options = patch_choices(browser["field"], size_widget.value)
        patch_widget.value = patch_widget.options[0][1]
        update_cells()


def change_field(*_):
    state = browser["state"]
    if state is None or field_widget.value is None:
        return
    show_button.disabled = True
    with status_output:
        clear_output(wait=True)
        print("Loading H&E and cell coordinates…")
        old = browser["field"]
        browser["field"] = None
        if old is not None and hasattr(old["tissue"], "store"):
            hest.close_wsi(old["tissue"])
        browser["field"] = load_field(state, field_widget.value)
        update_patches()
        show_button.disabled = False
        print("Ready. Choose a patch/cell and click Show selection.")


def on_load(_):
    show_button.disabled = save_button.disabled = True
    with status_output:
        clear_output(wait=True)
        print("Loading saved predictions and checking alignment…")
        state = load_dataset(dataset_widget.value)
        browser["last"] = None
        browser["state"] = state
        offered = [m for m in OFFERED_METHODS if m in state["methods"]]
        assert offered, f"{state['choice']} has no saved results for {OFFERED_METHODS}"
        method_widget.options = [(METHODS[m], m) for m in offered]
        method_widget.value = offered[0]
        # Clear the old selection before assigning options; observers load the new field.
        field_widget.unobserve(change_field, names="value")
        field_widget.options = []
        field_widget.options = [(f"{r.sample_id} · {r.evaluated_cells:,} evaluated cells", r.sample_id)
                                for r in state["fields"].itertuples()]
        field_widget.value = field_widget.options[0][1]
        field_widget.observe(change_field, names="value")
        change_field()
    with catalog_output:
        clear_output(wait=True)
        display(Markdown(f"**Loaded {state['choice']} · CellBridge**. All saved method metrics checked. "
                         "Choose any field; accuracy is shown for context, not used to rank fields."))
        show_table(state["fields"])
        metrics = state["metrics"].loc[state["methods"],
            ["target_coverage", "marginal_coverage", "mean_prediction_set_size", "singleton_rate"]].reset_index()
        show_table(metrics.rename(columns={"marginal_coverage": "full-test original-set coverage"}))
    with plot_output:
        clear_output()


def on_show(_):
    with plot_output:
        clear_output(wait=True)
        save_button.disabled = True
        state, field = browser["state"], browser["field"]
        cell_id, bounds = None, patch_widget.value
        if mode_widget.value == "Cell":
            cell_id = browser["cell_ids"].get(cell_widget.value)
            if cell_id is None:
                print("Select a cell from the suggestions first.")
                return
            row = field["cells"].set_index("row_id").loc[cell_id]
            height, width = field["tissue"].shape[:2]
            size = size_widget.value
            x0, y0 = max(0, round(row.x) - size // 2), max(0, round(row.y) - size // 2)
            bounds = (x0, y0, min(width, x0 + size), min(height, y0 + size))
        browser["last"] = render_selection(state, field, method_widget.value, bounds, cell_id, smooth_widget.value)
        save_button.disabled = False


def on_save(_):
    from datetime import datetime, timezone
    state, last = browser["state"], browser["last"]
    if last is None:
        return
    directory = EXPORT_ROOT / state["dataset"] / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    directory.mkdir(parents=True, exist_ok=False)
    last["figure"].savefig(directory / "view.png", dpi=180, bbox_inches="tight")
    last["cells"].to_csv(directory / "cells.csv", index=False)
    provenance = {k: v for k, v in last.items() if k not in {"figure", "cells"}}
    provenance.update(model="CellBridge", representation="layers_4_cells_all/lora_adapted", task="all", probe="linear",
        training_config=str(TRAIN_CONFIG), trainable_blocks=training["model"]["trainable_blocks"],
        choice=state["choice"], predictions=str(state["directory"] / "predictions.npz"),
        mapping=str(state["mapping_path"]), ontology=state["config"]["conformal"]["ontology"],
        ontology_release=state["ontology"].release, ontology_sha256=state["ontology"].sha256,
        nominal_coverage=float(state["metrics"].loc[last["method"], "target_coverage"]),
        depth_definition="Shortest true-term-to-LCA distance; excluded/empty sets have no depth",
        lca_rule="Lowest candidates; fewest descendant classes, greatest root depth, then ID")
    (directory / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
    with status_output:
        clear_output(wait=True)
        print("Saved displayed view, cell table and provenance to", directory)


load_button.on_click(on_load)
show_button.on_click(on_show)
save_button.on_click(on_save)
field_widget.observe(change_field, names="value")
size_widget.observe(update_patches, names="value")
patch_widget.observe(update_cells, names="value")
whole_field_widget.observe(update_cells, names="value")
cell_search.observe(update_cells, names="value")
display(widgets.VBox([widgets.HBox([dataset_widget, load_button]), status_output, catalog_output,
    field_widget, widgets.HBox([size_widget, mode_widget]), patch_widget,
    whole_field_widget, cell_search, cell_count, cell_widget, widgets.HBox([method_widget, smooth_widget]),
    widgets.HBox([show_button, save_button]), plot_output]))
on_load(None)
on_show(None)

## Tissue-level view of a HEST section

The explorer above starts from one patch. This section starts from the whole slide: every grid patch of a HEST section is reduced to one number, painted over the H&E, and can then be opened in the same four-panel detail view.

1. Choose a **section**, a **statistic**, a **patch size** and the minimum number of evaluated cells a patch must hold, then click **Show tissue map**. The left panel is the H&E; the right panel paints each patch by the chosen statistic. **Red is always the worse end.**
2. **Hover any painted patch** to read its name, its pixel bounds, its evaluated-cell count and its value.
3. The **Patch** menu lists the same patches worst first, with the same names. Pick one — typing a name from the map filters the list — and click **Open selected patch** to render it above the notes, with its cell table and ontology paths.

| Statistic | Per patch, over its evaluated cells |
|---|---|
| Mean / median depth · coarseness | Steps up from the reference term to the LCA, over **ancestor-covered cells only**. Excluded cells carry no depth and are counted by the next row instead. |
| Mean prediction-set size | Mean number of candidate labels in the selected method's saved set. |
| Reference excluded by ancestor | Fraction of cells whose reference label falls outside the ancestor-expanded set — the black **×** of the detail view. |
| Empty prediction sets | Fraction of cells with an empty saved set — the red **+**. |
| Ancestor-expanded / original-set coverage | Fraction of cells whose reference label is in the expanded set, or in the saved set itself. |
| Top-1 accuracy | Fraction of cells whose highest-scoring label equals the reference. |

Patches use the grid of the patch menu above (`x // size`, `y // size` in level-0 H&E pixels), so a 512 px tissue patch is one of that menu's patches. A patch is painted only when it holds at least the chosen number of evaluated cells; the rest of the tissue is left plain. **Only the held-out test region of a section carries evaluated cells**, so the painted area is usually a part of the slide rather than all of it. Patch size is given in level-0 pixels, whose physical size differs between sections; the figure states the µm equivalent.

Depth keeps the fixed ontology scale of the detail view, so depth maps are comparable between sections, methods and patch sizes. Every other statistic is stretched to the 2nd–98th percentile of the painted patches, which makes its pattern readable but comparable only within that map; its colour-bar label says so. A patch mean over a few dozen cells is noisy — raise **Min cells** before reading a pattern into it, and remember that the saved nominal coverage is a population-level target, not a per-patch guarantee.

The overview reads a low-resolution pyramid level of the whole slide, while the detail view still reads full-resolution pixels. Changing a control does not redraw the map; click **Show tissue map** again. **Open selected patch** renders the chosen patch without redrawing the map, and the TERRA section below projects that same patch.

In [ ]:
TISSUE_STATISTICS = {
    # label: patch column, higher is better, keeps the fixed ontology-depth scale
    "Mean depth / coarseness": ("mean_depth", False, True),
    "Median depth / coarseness": ("median_depth", False, True),
    "Mean prediction-set size": ("mean_set_size", False, False),
    "Reference excluded by ancestor": ("excluded_fraction", False, False),
    "Empty prediction sets": ("empty_fraction", False, False),
    "Ancestor-expanded coverage": ("ancestor_coverage", True, False),
    "Original-set coverage": ("original_coverage", True, False),
    "Top-1 accuracy": ("top1_accuracy", True, False),
}
HEST_CHOICES = [choice for choice in catalog.index if catalog.loc[choice, "dataset"] == "hest"]
assert HEST_CHOICES, f"No HEST sections with saved cell-level predictions under {MODEL_RESULTS}"


def section_thumbnail(field, target_width=1600):
    """Read the smallest whole-slide pyramid level at least this many pixels wide."""
    import tifffile
    height, width = field["tissue"].shape[:2]
    with tifffile.TiffFile(field["image_path"]) as handle:
        levels = handle.series[0].levels
        level = int(np.clip(np.floor(np.log2(max(height, width) / target_width)), 0, len(levels) - 1))
        return np.asarray(levels[level].asarray())[..., :3]


def patch_statistics(rows, size, shape, min_cells):
    """Summarize the evaluated cells of every grid patch of one section."""
    height, width = shape
    cells = rows.loc[rows.evaluated].copy()
    cells["row"], cells["column"] = (cells.y // size).astype(int), (cells.x // size).astype(int)
    cells["ancestor_covered"] = cells.ancestor_covered.astype(bool)
    cells["raw_covered"] = cells.raw_covered.astype(bool)
    cells["excluded"] = cells.set_size.gt(0) & ~cells.ancestor_covered
    cells["empty"] = cells.set_size.eq(0)
    cells["correct"] = cells.truth.eq(cells.predicted)
    # Excluded cells carry no depth, so patch depth averages the ancestor-covered cells only.
    table = cells.groupby(["row", "column"]).agg(
        evaluated_cells=("row_id", "size"), mean_depth=("depth", "mean"),
        median_depth=("depth", "median"), mean_set_size=("set_size", "mean"),
        excluded_fraction=("excluded", "mean"), empty_fraction=("empty", "mean"),
        ancestor_coverage=("ancestor_covered", "mean"), original_coverage=("raw_covered", "mean"),
        top1_accuracy=("correct", "mean")).reset_index()
    table.insert(0, "patch", [f"r{r:03d}c{c:03d}" for r, c in zip(table.row, table.column)])
    table["x0"], table["y0"] = table.column * size, table.row * size
    table["x1"] = np.minimum(table.x0 + size, width)
    table["y1"] = np.minimum(table.y0 + size, height)
    return table.loc[table.evaluated_cells >= min_cells].reset_index(drop=True)


def colour_scale(state, statistic, values):
    """Depth keeps the ontology's fixed scale; other statistics stretch to this section."""
    _, higher_is_better, fixed = TISSUE_STATISTICS[statistic]
    cmap = DEPTH_CMAP.reversed() if higher_is_better else DEPTH_CMAP
    if fixed:
        return cmap, 0.0, max(max(d.values()) - 1 for d in state["graph"].distances), statistic
    low, high = np.nanpercentile(values, [2, 98])
    return cmap, float(low), float(max(high, low + 1e-6)), f"{statistic} · scaled to this section"


# Notebook viewers do not all draw the browser's own title tooltip, so every patch carries
# a hover label of its own; title stays on as a fallback for plain HTML exports.
HOVER_STYLE = """<style>
#{name} {{position:relative;display:inline-block;max-width:100%}}
#{name} > img {{width:100%;display:block}}
#{name} .hit {{position:absolute;cursor:help}}
#{name} .hit:hover {{outline:1.5px solid #111;background:rgba(255,255,255,.25);z-index:2}}
#{name} .tip {{display:none;position:absolute;z-index:3;white-space:nowrap;pointer-events:none;
 padding:4px 7px;border-radius:4px;background:rgba(20,20,20,.93);color:#fff;
 font:11px/1.35 -apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
 box-shadow:0 1px 4px rgba(0,0,0,.4)}}
#{name} .hit:hover .tip {{display:block}}
#{name} .from-left .tip {{left:0}}
#{name} .from-right .tip {{right:0}}
#{name} .below .tip {{top:100%;margin-top:4px}}
#{name} .above .tip {{bottom:100%;margin-bottom:4px}}
</style>"""


def hover_targets(fig, ax, table, tips):
    """Display the drawn figure with one hover label per patch."""
    import base64
    import html
    import io
    import uuid
    fig.canvas.draw()
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=fig.dpi)
    width, height = fig.get_size_inches() * fig.dpi
    near = ax.transData.transform(np.column_stack([table.x0, table.y0]))
    far = ax.transData.transform(np.column_stack([table.x1, table.y1]))
    name = f"tissue-map-{uuid.uuid4().hex[:8]}"
    boxes = []
    for (left, top), (right, bottom), tip in zip(near, far, tips):
        # Open the label away from whichever edge of the figure the patch sits against.
        anchor = "from-right" if left / width > .55 else "from-left"
        side = "above" if top / height < .45 else "below"
        label = html.escape(str(tip))
        boxes.append(f'<div class="hit {anchor} {side}" title="{label}" '
                     f'style="left:{left / width:.4%};top:{1 - top / height:.4%};'
                     f'width:{(right - left) / width:.4%};height:{(top - bottom) / height:.4%}">'
                     f'<span class="tip">{label}</span></div>')
    display(HTML(HOVER_STYLE.format(name=name) + f'<div id="{name}">'
                 '<img src="data:image/png;base64,'
                 f'{base64.b64encode(buffer.getvalue()).decode()}">'
                 + "".join(boxes) + '</div>'))


def render_section(state, field, table, size, statistic, method, min_cells):
    column = TISSUE_STATISTICS[statistic][0]
    cmap, low, high, bar_label = colour_scale(state, statistic, table[column])
    height, width = field["tissue"].shape[:2]
    grid = np.full((int(np.ceil(height / size)), int(np.ceil(width / size))), np.nan)
    grid[table.row.to_numpy(), table.column.to_numpy()] = table[column].to_numpy(float)
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), layout="constrained")
    thumbnail = section_thumbnail(field)
    for ax in axes:
        ax.imshow(thumbnail, extent=(0, width, height, 0), alpha=1 if ax is axes[0] else .5)
        ax.set(xlim=(0, width), ylim=(height, 0))
        ax.axis("off")
    axes[0].set_title("H&E · whole section")
    axes[1].set_title(f"{statistic} · {size} px patches")
    image = axes[1].imshow(grid, extent=(0, grid.shape[1] * size, grid.shape[0] * size, 0),
                           cmap=cmap, vmin=low, vmax=high, alpha=.8, interpolation="nearest")
    fig.colorbar(image, ax=axes[1], shrink=.7, label=bar_label)
    scale_um = 10 ** np.floor(np.log10(width * field["mpp"] / 4))
    bar_x, bar_y = .04 * width, .97 * height
    axes[0].plot([bar_x, bar_x + scale_um / field["mpp"]], [bar_y, bar_y], c="black", lw=3)
    axes[0].text(bar_x, bar_y - .015 * height, f"{scale_um:g} µm",
                 bbox=dict(facecolor="white", alpha=.8, edgecolor="none"))
    fig.suptitle(f"{state['choice']} · CellBridge · {METHODS[method]}\n"
                 f"{len(table):,} patches with at least {min_cells} evaluated cells · "
                 f"{size} px ≈ {size * field['mpp']:.0f} µm", fontsize=11)
    tips = [f"{patch} · x={x0}:{x1}, y={y0}:{y1} · {count:,} evaluated cells · {statistic}: {value:.2f}"
            for patch, x0, x1, y0, y1, count, value in
            zip(table.patch, table.x0, table.x1, table.y0, table.y1, table.evaluated_cells, table[column])]
    hover_targets(fig, axes[1], table, tips)
    plt.close(fig)


section_widget = widgets.Dropdown(options=HEST_CHOICES, description="Section:",
                                  layout=widgets.Layout(width="450px"))
statistic_widget = widgets.Dropdown(options=list(TISSUE_STATISTICS), description="Statistic:",
                                    layout=widgets.Layout(width="450px"))
tissue_size_widget = widgets.Dropdown(options=[512, 1024, 2048], value=1024, description="Patch (px):")
min_cells_widget = widgets.Dropdown(options=[1, 10, 25, 50, 100], value=25, description="Min cells:")
tissue_method_widget = widgets.Dropdown(options=[(METHODS[m], m) for m in OFFERED_METHODS],
                                        value=OFFERED_METHODS[0], description="Method:",
                                        layout=widgets.Layout(width="350px"))
tissue_smooth_widget = widgets.Checkbox(value=False, description="Smooth depth map")
tissue_button = widgets.Button(description="Show tissue map", button_style="primary")
tissue_patch_widget = widgets.Combobox(options=[], ensure_option=True, description="Patch:",
                                       layout=widgets.Layout(width="95%"))
patch_button = widgets.Button(description="Open selected patch", disabled=True)
tissue_status, tissue_output, patch_output = widgets.Output(), widgets.Output(), widgets.Output()
tissue = {"choice": None, "state": None, "field": None, "rows": {}, "patches": {}}


def tissue_section_data(choice):
    """Load one HEST section, keeping only the current whole-slide image open."""
    if tissue["choice"] != choice:
        if tissue["field"] is not None:
            hest.close_wsi(tissue["field"]["tissue"])
        tissue.update(choice=None, state=None, field=None, rows={}, patches={})
        state = load_dataset(choice)
        tissue.update(choice=choice, state=state,
                      field=load_field(state, catalog.loc[choice, "section"]))
    return tissue["state"], tissue["field"]


def on_tissue_show(_):
    patch_button.disabled = True
    size, min_cells, statistic = tissue_size_widget.value, min_cells_widget.value, statistic_widget.value
    with tissue_status:
        clear_output(wait=True)
        print("Loading the section and scoring its cells…")
        state, field = tissue_section_data(section_widget.value)
        offered = [m for m in OFFERED_METHODS if m in state["methods"]]
        assert offered, f"{state['choice']} has no saved results for {OFFERED_METHODS}"
        tissue_method_widget.options = [(METHODS[m], m) for m in offered]
        method = tissue_method_widget.value
        if method not in tissue["rows"]:
            tissue["rows"][method] = evaluate_field(state, field, method)
        rows = tissue["rows"][method]
        column, higher_is_better, _ = TISSUE_STATISTICS[statistic]
        table = patch_statistics(rows, size, field["tissue"].shape[:2], min_cells)
        table = table.sort_values(column, ascending=higher_is_better, na_position="last")
        tissue["patches"] = {
            f"{r.patch} | x={r.x0}, y={r.y0} | {r.evaluated_cells:,} cells | "
            f"{statistic}: {getattr(r, column):.2f}": (r.x0, r.y0, r.x1, r.y1)
            for r in table.itertuples()}
        tissue_patch_widget.options = list(tissue["patches"])
        if tissue_patch_widget.options:
            tissue_patch_widget.value = tissue_patch_widget.options[0]
            patch_button.disabled = False
        print("Ready. Patches are listed worst first; pick one and open it.")
    with patch_output:
        clear_output()
    with tissue_output:
        clear_output(wait=True)
        evaluated = rows.loc[rows.evaluated]
        nominal = float(state["metrics"].loc[method, "target_coverage"])
        display(Markdown(f"**Whole section: {len(evaluated):,} evaluated cells** among "
                         f"{len(rows):,} annotated cells. Saved nominal coverage: {nominal:.0%}."))
        show_table(pd.DataFrame([{
            "top-1 correct": evaluated.truth.eq(evaluated.predicted).mean(),
            "original-set coverage": evaluated.raw_covered.astype(bool).mean(),
            "ancestor-expanded coverage": evaluated.ancestor_covered.astype(bool).mean(),
            "mean set size": evaluated.set_size.mean(),
            "mean depth (covered only)": evaluated.depth.mean(),
            "empty sets": int(evaluated.set_size.eq(0).sum())}]))
        if table.empty:
            display(Markdown(f"**No patch holds {min_cells} evaluated cells at {size} px.** "
                             "Lower either control and show the map again."))
            return
        render_section(state, field, table, size, statistic, method, min_cells)
        display(Markdown("Hover any patch on the map for its name, bounds and value. "
                         f"**Patches, worst first by {statistic}:**"))
        show_table(table.drop(columns=["row", "column"]))


def on_patch_open(_):
    with patch_output:
        clear_output(wait=True)
        bounds = tissue["patches"].get(tissue_patch_widget.value)
        if bounds is None:
            print("Choose a patch from the suggestions first.")
            return
        render_selection(tissue["state"], tissue["field"], tissue_method_widget.value, bounds,
                         None, tissue_smooth_widget.value)


tissue_button.on_click(on_tissue_show)
patch_button.on_click(on_patch_open)
display(widgets.VBox([widgets.HBox([section_widget, tissue_button]),
    widgets.HBox([statistic_widget, tissue_size_widget, min_cells_widget]),
    widgets.HBox([tissue_method_widget, tissue_smooth_widget]), tissue_status, tissue_output,
    tissue_patch_widget, patch_button, patch_output]))
on_tissue_show(None)

## Cross-check a patch against TERRA transcriptomic embeddings

CellBridge reads H&E only. The HEST sections also carry **TERRA** embeddings, saved during preprocessing: for every cell, a 384-dimensional representation of its own Xenium transcript counts (*cell embedding*) and one of its spatial transcriptomic neighbourhood (*neighbourhood embedding*). This section takes the patch currently chosen above and asks one question: **do the cells CellBridge is unsure about also sit in ambiguous places of the transcriptomic space?**

1. Show a tissue map above and pick a **Patch**. This section always uses that patch, that section and the same method.
2. Choose the **Embedding**, what to **Colour by**, and how many **Context** cells to project alongside the patch. Click **Fetch TERRA embeddings**. Nothing is computed until you click; 20,000 context cells take a minute or two and 50,000 several minutes, because the projection is seeded for reproducibility and therefore single-threaded.
3. Cells with no saved test prediction stay as hollow grey markers, as everywhere else in this notebook; the black **×** and red **+** keep their meanings.

| Panel | Reads as |
|---|---|
| UMAP | PCA to 50 components, then UMAP (`n_neighbors=15`, `min_dist=0.1`, `random_state=0`) over the patch cells plus a uniform random sample of the section's other cells. The patch is in colour, the context in light grey. Axes have no units; only neighbourhoods mean anything, and distances between separated groups do not. |
| The same patch in tissue | The same cells and colours over the H&E crop, so a group seen in the UMAP can be located in the tissue. |
| Neighbourhood agreement | For each evaluated patch cell, the share of its 15 nearest cells **in TERRA PCA space** carrying the same reference label, averaged within each depth level. Black is the excluded group, red the empty sets, and `n` is the number of cells behind each bar. |

**Read this as support, not as validation.** The HEST reference labels are themselves derived from the transcript counts TERRA embeds, so high agreement at depth 0 is expected and is not independent evidence that a label is right. What is informative is the contrast: whether CellBridge's H&E-derived uncertainty — larger depth, an excluded reference, an empty set — falls on cells whose transcriptomic neighbourhood is also mixed. Where it does, the coarser prediction reflects ambiguity that is present in the tissue; where agreement stays high while depth grows, the uncertainty is a limitation of the H&E view rather than of the cell.

Context cells are drawn uniformly from the whole section, including cells with no test prediction, so the grey backdrop shows the section's transcriptomic landscape rather than only its evaluated part. Agreement is measured among the evaluated cells of the projected sample, so a larger context makes it less dependent on the patch's own composition. UMAP is fitted afresh on every click: the layout changes when the embedding or the context size changes, and only structure within one figure should be compared. A patch of a few dozen cells supports no claim about the section.

In [ ]:
TERRA_MATRICES = {"Cell embedding": "cell_emb", "Neighbourhood embedding": "neighborhood_emb"}
TERRA_COLOURS = {
    # label: cell column, discrete classes
    "Reference label": ("truth", True),
    "CellBridge predicted label": ("predicted", True),
    "Ontology depth / coarseness": ("depth", False),
    "Prediction-set size": ("set_size", False),
}
TERRA_NEIGHBOURS = 15
TERRA_CACHE = {}


def terra_file(state):
    """Locate one section's saved TERRA embeddings, resolved as the HEST loader does."""
    data = state["config"]["data"]
    manifest = pd.read_csv(data["manifest"])
    matches = manifest.loc[manifest["id"].map(hest._identifier).eq(state["section"])]
    assert len(matches) == 1, f"Expected one manifest row for {state['section']}"
    directory = matches.iloc[0].get("output_dir")
    directory = Path(data["root"]) / state["section"] if pd.isna(directory) else Path(directory)
    return directory / f"{state['section']}_terra.npz"


def terra_matrix(state, name):
    """Read one TERRA matrix; its rows follow the section's cell order."""
    key = (state["section"], name)
    if key not in TERRA_CACHE:
        TERRA_CACHE.clear()  # a section-sized matrix is a few hundred MB; keep only the current one
        with np.load(terra_file(state), allow_pickle=False) as archive:
            ids = np.asarray([hest._identifier(value) for value in archive["cell_ids"]])
            np.testing.assert_array_equal(ids, state["sample"].cell_ids)
            TERRA_CACHE[key] = np.ascontiguousarray(archive[name], dtype=np.float32)
    return TERRA_CACHE[key]


def terra_pool(state, rows, matrix, positions, context, seed=0):
    """Project the patch cells, then a uniform random sample of the rest of the section."""
    import umap
    from sklearn.decomposition import PCA
    others = np.setdiff1d(np.arange(len(matrix)), positions)
    if context < len(others):
        others = np.sort(np.random.default_rng(seed).choice(others, context, replace=False))
    order = np.concatenate([positions, others])
    assert len(order) >= 10, "Fewer than ten cells to project; take a larger patch or add context."
    values = matrix[order]
    reduced = PCA(n_components=min(50, *values.shape), random_state=seed).fit_transform(values)
    points = np.asarray(umap.UMAP(n_neighbors=max(2, min(15, len(order) - 1)), min_dist=.1,
                                  random_state=seed).fit_transform(reduced))
    pool = pd.DataFrame({"row_id": np.asarray(state["sample"].cell_ids)[order],
                         "in_patch": np.arange(len(order)) < len(positions),
                         "umap_x": points[:, 0], "umap_y": points[:, 1]})
    pool = pool.merge(rows, on="row_id", how="left", validate="one_to_one")
    pool["evaluated"] = pool.evaluated.fillna(False).astype(bool)
    return pool, reduced


def terra_agreement(reduced, pool, neighbours=TERRA_NEIGHBOURS):
    """Share of a cell's nearest TERRA neighbours carrying its own reference label."""
    from sklearn.neighbors import NearestNeighbors
    known = np.flatnonzero(pool.evaluated.to_numpy())
    shares = np.full(len(pool), np.nan)
    if len(known) > 1:
        count = min(neighbours, len(known) - 1)
        found = NearestNeighbors(n_neighbors=count + 1).fit(reduced[known]).kneighbors(
            reduced[known], return_distance=False)
        truth = pool.truth.to_numpy()[known]
        # A point is normally its own first neighbour; drop it wherever it is returned.
        elsewhere = found != np.arange(len(known))[:, None]
        shares[known] = ((truth[found] == truth[:, None]) & elsewhere).sum(1) / elsewhere.sum(1)
    return shares


def render_terra(state, field, rows, bounds, method, name, colour, context):
    """Project one patch into TERRA space beside its tissue and its neighbourhood agreement."""
    matrix = terra_matrix(state, TERRA_MATRICES[name])
    patch = cells_in_bounds(rows, bounds)
    positions = pd.Index(state["sample"].cell_ids).get_indexer(patch.row_id)
    assert (positions >= 0).all(), "A patch cell is absent from the TERRA embedding"
    pool, reduced = terra_pool(state, rows, matrix, positions, context)
    pool["neighbour_agreement"] = terra_agreement(reduced, pool)
    column, discrete = TERRA_COLOURS[colour]
    classes = state["saved"]["classes"].astype(str)
    maximum = max(max(distances.values()) - 1 for distances in state["graph"].distances)
    inside, outside = pool.loc[pool.in_patch], pool.loc[~pool.in_patch]
    scored = inside.loc[inside.evaluated]
    x0, y0, x1, y1 = map(int, bounds)
    tissue = np.asarray(field["tissue"][y0:y1, x0:x1])[..., :3]
    fig, axes = plt.subplots(1, 3, figsize=(19, 6.4), layout="constrained")
    axes[0].scatter(outside.umap_x, outside.umap_y, s=3, c="#dcdcdc", linewidths=0)
    axes[0].set(xlabel="UMAP 1", ylabel="UMAP 2", xticks=[], yticks=[])
    axes[1].imshow(tissue, extent=(x0 - .5, x1 - .5, y1 - .5, y0 - .5), alpha=.65)
    axes[1].set(xlim=(x0 - .5, x1 - .5), ylim=(y1 - .5, y0 - .5))
    axes[1].axis("off")
    blank = inside.loc[~inside.evaluated]
    axes[0].scatter(blank.umap_x, blank.umap_y, s=24, facecolors="none", edgecolors="grey", linewidths=.5)
    axes[1].scatter(blank.x, blank.y, s=44, facecolors="none", edgecolors="grey", linewidths=.5)
    if discrete:
        colours = dict(zip(classes, plt.get_cmap("tab20")(np.linspace(0, 1, len(classes)))))
        for label in classes:
            group = scored.loc[scored[column].eq(label)]
            axes[0].scatter(group.umap_x, group.umap_y, s=32, color=colours[label],
                            edgecolors="white", linewidths=.4)
            axes[1].scatter(group.x, group.y, s=52, color=colours[label],
                            edgecolors="white", linewidths=.4)
        fig.legend(handles=[Line2D([], [], marker="o", linestyle="none", color=colours[c], label=c)
                            for c in classes if scored[column].eq(c).any()],
                   loc="outside lower center", ncol=5, fontsize=8)
    else:
        if column == "depth":
            cmap = ListedColormap(DEPTH_CMAP(np.linspace(0, 1, maximum + 1)))
            norm, ticks = BoundaryNorm(np.arange(-.5, maximum + 1.5), cmap.N), range(maximum + 1)
            marked = scored.loc[scored.depth.notna()]
        else:
            top = int(scored.set_size.max()) if len(scored) else 1
            cmap, norm, ticks, marked = DEPTH_CMAP, plt.Normalize(0, max(top, 1)), None, scored
        axes[0].scatter(marked.umap_x, marked.umap_y, c=marked[column], cmap=cmap, norm=norm,
                        s=32, edgecolors="white", linewidths=.4)
        axes[1].scatter(marked.x, marked.y, c=marked[column], cmap=cmap, norm=norm,
                        s=52, edgecolors="white", linewidths=.4)
        fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=axes[1], shrink=.75,
                     ticks=ticks, label=colour)
        if column == "depth":
            excluded = scored.loc[scored.depth.isna() & scored.set_size.gt(0)]
            empty = scored.loc[scored.set_size.eq(0)]
            axes[0].scatter(excluded.umap_x, excluded.umap_y, marker="x", c="black", s=44)
            axes[0].scatter(empty.umap_x, empty.umap_y, marker="+", c="#c62828", s=60)
            axes[1].scatter(excluded.x, excluded.y, marker="x", c="black", s=64)
            axes[1].scatter(empty.x, empty.y, marker="+", c="#c62828", s=88)
    if scored.neighbour_agreement.notna().any():
        # Depth levels keep the ontology ramp; excluded and empty sets keep their × and + colours.
        buckets = ["Empty set" if size == 0 else "Excluded" if pd.isna(depth) else str(int(depth))
                   for depth, size in zip(scored.depth, scored.set_size)]
        summary = scored.assign(bucket=buckets).groupby("bucket").agg(
            agreement=("neighbour_agreement", "mean"), cells=("row_id", "size"))
        summary = summary.reindex([b for b in [*map(str, range(maximum + 1)), "Excluded", "Empty set"]
                                   if b in summary.index])
        palette = {"Excluded": "black", "Empty set": "#c62828"}
        bars = axes[2].bar(summary.index, summary.agreement, edgecolor="white",
                           color=[palette[b] if b in palette
                                  else DEPTH_CMAP(int(b) / max(maximum, 1))
                                  for b in summary.index])
        axes[2].bar_label(bars, labels=[f"n={n:,}" for n in summary.cells], fontsize=8, padding=2)
        axes[2].set(ylim=(0, 1.15), xlabel="Ontology depth / coarseness",
                    ylabel=f"Share of the {TERRA_NEIGHBOURS} nearest TERRA neighbours\n"
                           "with the same reference label")
        axes[2].spines[["top", "right"]].set_visible(False)
    else:
        axes[2].text(.5, .5, "No evaluated cells to compare", ha="center", va="center",
                     transform=axes[2].transAxes)
        axes[2].axis("off")
    axes[0].set_title(f"TERRA {name.lower()} · UMAP\n"
                      f"{len(inside):,} patch cells over {len(outside):,} context cells")
    axes[1].set_title(f"The same patch in tissue\ncoloured by {colour.lower()}")
    axes[2].set_title(f"Transcriptomic neighbourhood agreement\n"
                      f"{TERRA_NEIGHBOURS} nearest neighbours in TERRA PCA space")
    fig.suptitle(f"{state['choice']} · TERRA {name.lower()} · CellBridge · {METHODS[method]}\n"
                 f"{field['sample_id']} · x={x0}:{x1}, y={y0}:{y1}", fontsize=11)
    display(fig)
    plt.close(fig)
    display(Markdown(f"**{len(inside):,} cells in the patch; {len(scored):,} with test predictions**, "
                     f"projected with {len(outside):,} context cells from the rest of the section. "
                     "TERRA embeds these cells' Xenium counts; CellBridge never saw them."))
    if len(scored):
        show_table(pd.DataFrame([{
            "mean neighbour agreement": scored.neighbour_agreement.mean(),
            "mean depth (covered only)": scored.depth.mean(),
            "ancestor-expanded coverage": scored.ancestor_covered.astype(bool).mean(),
            "mean set size": scored.set_size.mean(),
            "reference excluded": int((scored.depth.isna() & scored.set_size.gt(0)).sum()),
            "empty sets": int(scored.set_size.eq(0).sum())}]))
        display(Markdown("**Patch cells, least transcriptomically typical first** — the cells whose "
                         "TERRA neighbours least often carry their own reference label:"))
        show_table(scored.sort_values("neighbour_agreement")[
            ["number", "row_id", "x", "y", "label", "truth", "predicted", "prediction_set",
             "set_size", "ancestor", "depth", "neighbour_agreement"]])
    return dict(pool=pool, patch=scored, bounds=[x0, y0, x1, y1], method=method,
                embedding=TERRA_MATRICES[name], colour=column, context=len(outside))


terra_embedding_widget = widgets.Dropdown(options=list(TERRA_MATRICES), description="Embedding:",
                                          layout=widgets.Layout(width="420px"))
terra_colour_widget = widgets.Dropdown(options=list(TERRA_COLOURS), value="Ontology depth / coarseness",
                                       description="Colour by:", layout=widgets.Layout(width="420px"))
terra_context_widget = widgets.Dropdown(
    options=[("Patch only", 0), ("5,000 cells", 5000), ("20,000 cells", 20000), ("50,000 cells", 50000)],
    value=5000, description="Context:")
terra_button = widgets.Button(description="Fetch TERRA embeddings", button_style="primary",
                              layout=widgets.Layout(width="220px"))
terra_status, terra_output = widgets.Output(), widgets.Output()
terra = {"last": None}


def on_terra_fetch(_):
    state, field = tissue["state"], tissue["field"]
    bounds = tissue["patches"].get(tissue_patch_widget.value)
    with terra_status:
        clear_output(wait=True)
        if state is None or field is None or bounds is None:
            print("Show a tissue map above and choose a patch first.")
            return
        assert state["dataset"] == "hest", "TERRA embeddings are saved for HEST sections only"
        print(f"Reading the TERRA {terra_embedding_widget.value.lower()} of {state['choice']} and "
              "projecting; a large context takes a few minutes…")
    method = tissue_method_widget.value
    if method not in tissue["rows"]:
        tissue["rows"][method] = evaluate_field(state, field, method)
    with terra_output:
        clear_output(wait=True)
        terra["last"] = render_terra(state, field, tissue["rows"][method], bounds, method,
                                     terra_embedding_widget.value, terra_colour_widget.value,
                                     terra_context_widget.value)
    with terra_status:
        clear_output(wait=True)
        print(f"Projected patch {tissue_patch_widget.value.split(' | ')[0]} · "
              f"{terra_embedding_widget.value.lower()} · "
              f"{terra['last']['context']:,} context cells.")


terra_button.on_click(on_terra_fetch)
display(widgets.VBox([widgets.HBox([terra_embedding_widget, terra_colour_widget]),
    widgets.HBox([terra_context_widget, terra_button]), terra_status, terra_output]))
print("Choose a patch in the tissue section above, then click Fetch TERRA embeddings.")

## Notes for your biological review

For a selected cell, compare the reference annotation with its morphology and neighbouring cells. Does the top prediction look plausible? If it is wrong, does the original candidate set still contain the reference? Does the common ancestor retain a useful biological identity, or become too broad? Inspect excluded cells alongside covered cells with large depth; the two represent different limitations. For a HEST patch, the TERRA section shows whether an uncertain cell is also ambiguous transcriptomically before you conclude that CellBridge is simply wrong.

For reproducibility, save the displayed view and record why you chose it. A favourable HEST example should be labelled as such if later selected by performance. HEST reference labels come from the prepared molecular annotations and are aligned to H&E; neighbouring cells without saved test predictions remain visible but are not scored. The displayed context crop is for review and need not equal the model's input crop.

**Technical references within this project:** [notebook 08](08_conformal_prediction.ipynb), the pinned mappings in `configs/preprocess/cell_ontology/`, and `cellbridge.inference.conformal.graph`. The LCA and spatial depth conventions follow the supplied `plot_ancestor_coarseness.py` and `plot_phenocell_uncertainty.py`; no code is imported from those scratch scripts. The notebook recomputes the LCA of each saved set, which can differ from a graph method's stored anchor when several ontology nodes have the same descendant classes.